In [ ]:
import numpy as np
import pandas as pd
import joblib
import logging

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from database import init_db, populate_sample_data, get_training_data


# -----------------------------
# 1. Load churn data from SQLite database
# -----------------------------

In [3]:
logging.basicConfig(level=logging.INFO)

In [5]:
logging.basicConfig(level=logging.INFO)

init_db()
populate_sample_data()

data = get_training_data()
logging.info("Training data loaded from SQLite database successfully")

if data.empty:
    raise ValueError("No training data available in the database. Check that sample data has been seeded.")


INFO:root:Risk scores generated successfully
INFO:root:Synthetic dataset created successfully


In [6]:
data.head(10)
data.to_csv("training_data_from_db.csv", index=False)
logging.info("Training data exported from SQLite to training_data_from_db.csv")


INFO:root:Synthetic dataset saved successfully


# -----------------------------
# 2. Split features and target
# -----------------------------

In [7]:
X = data[["age", "income", "tenure_months"]]
y = data["churn"]
logging.info("Features and target variable separated successfully")


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
logging.info("Data split into training and testing sets successfully")

INFO:root:Features and target variable separated successfully
INFO:root:Data split into training and testing sets successfully


# -----------------------------
# 3. Build model pipeline
# -----------------------------

In [8]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression())
])
logging.info("Model pipeline created successfully")

INFO:root:Model pipeline created successfully


# -----------------------------
# 4. Train model
# -----------------------------

In [9]:
model.fit(X_train, y_train)
logging.info("Model training completed successfully")

INFO:root:Model training completed successfully


# -----------------------------
# 5. Evaluate model
# -----------------------------

In [6]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.8833333333333333

              precision    recall  f1-score   support

           0       0.89      0.98      0.93       254
           1       0.79      0.33      0.46        46

    accuracy                           0.88       300
   macro avg       0.84      0.66      0.70       300
weighted avg       0.87      0.88      0.86       300



# -----------------------------
# 6. Save model
# -----------------------------

In [ ]:
joblib.dump(model, "churn_model.pkl")
logging.info("Model saved as churn_model.pkl")

batch_probs = model.predict_proba(X)[:, 1]
results = X.copy()
results["churn_probability"] = batch_probs
results["risk_band"] = results["churn_probability"].apply(lambda p: "high" if p >= 0.7 else "medium" if p >= 0.4 else "low")
results.to_csv("batch_predictions.csv", index=False)
logging.info("Batch predictions saved to batch_predictions.csv")

print("Model saved as churn_model.pkl")
print("Batch predictions saved as batch_predictions.csv")


Model saved as churn_model.pkl
